# Imports

In [9]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import MulticlassThresholdOptimizer, load_pickle

from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [ ]:
label_encoder = load_pickle('../models/')

# Loading Datasets

In [2]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_one.parquet')

In [3]:
X_train.head()

,lgbm_0,lgbm_1,cat_0,cat_1,xgb_0,xgb_1,hist_0,hist_1,extra_0,extra_1,rf_0,rf_1
0,9.999566e-01,0.000043,0.999900,0.000050,0.999734,0.000192,9.998896e-01,0.000105,0.912643,0.007039,0.999665,0.000155
1,9.895666e-01,0.000366,0.986604,0.000093,0.985616,0.000664,9.819315e-01,0.000384,0.803503,0.004824,0.965267,0.000184
2,3.890271e-07,1.000000,0.000016,0.999984,0.000088,0.999889,3.930889e-07,1.000000,0.006700,0.965519,0.000113,0.999887
3,9.998353e-01,0.000164,0.999925,0.000071,0.999614,0.000265,9.991001e-01,0.000895,0.932451,0.006030,0.999310,0.000345
4,9.993004e-01,0.000692,0.999187,0.000766,0.998981,0.000832,9.846569e-01,0.015320,0.915246,0.008021,0.997708,0.000696


In [4]:
X_test.head()

,lgbm_0,lgbm_1,cat_0,cat_1,xgb_0,xgb_1,hist_0,hist_1,extra_0,extra_1,rf_0,rf_1
0,0.999271,0.000708,0.996756,0.002437,0.997384,0.002024,0.999265,0.000701,0.611632,0.076394,0.977833,0.009356
1,0.997605,0.002393,0.997894,0.002102,0.998017,0.001866,0.971958,0.028034,0.942311,0.018000,0.998781,0.000921
2,0.998315,0.000205,0.999515,0.000035,0.994165,0.001013,0.996011,0.000573,0.530537,0.015231,0.980001,0.002162
3,0.000593,0.000201,0.001054,0.000337,0.003027,0.001288,0.000174,0.000112,0.086899,0.066500,0.009974,0.007596
4,0.999900,0.000097,0.999687,0.000311,0.999482,0.000408,0.999597,0.000382,0.945084,0.015204,0.999542,0.000244


# Machine Learning

In [5]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000),
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx, :]
        X_valid_fold = X.iloc[valid_idx, :]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = LogisticRegression(**params)
        model.fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)

        score = log_loss(y_valid_fold, proba)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=20, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-04 16:18:02,789] A new study created in memory with name: no-name-03c3c641-3b99-49fc-8a38-2c30a0c0e120
Best trial: 5. Best value: 0.127569:   5%|██████▉                                                                                                                                    | 1/20 [00:18<05:57, 18.81s/it]

[I 2026-06-04 16:18:21,596] Trial 5 finished with value: 0.12756909329978816 and parameters: {'solver': 'saga', 'C': 0.00406966840233501, 'l1_ratio': 0.892836984648035, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.006704822604034238, 'max_iter': 2601}. Best is trial 5 with value: 0.12756909329978816.


Best trial: 5. Best value: 0.127569:  10%|█████████████▉                                                                                                                             | 2/20 [00:25<03:28, 11.56s/it]

[I 2026-06-04 16:18:28,086] Trial 3 finished with value: 0.14534839859392362 and parameters: {'solver': 'saga', 'C': 0.7989892100430462, 'l1_ratio': 0.922830863658472, 'class_weight': None, 'fit_intercept': False, 'tol': 0.008461246664135514, 'max_iter': 1982}. Best is trial 5 with value: 0.12756909329978816.


Best trial: 8. Best value: 0.127472:  15%|████████████████████▊                                                                                                                      | 3/20 [00:29<02:18,  8.15s/it]

...

[I 2026-06-04 16:22:49,319] Trial 10 pruned. 


Best trial: 4. Best value: 0.102409:  95%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏      | 19/20 [15:48<04:05, 245.27s/it]

[I 2026-06-04 16:33:51,363] Trial 2 finished with value: 0.12746959133963573 and parameters: {'solver': 'saga', 'C': 66.24264330978316, 'l1_ratio': 0.8402477394168169, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.00020344872835997542, 'max_iter': 2746}. Best is trial 4 with value: 0.1024090857320655.


Best trial: 4. Best value: 0.102409: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [16:45<00:00, 50.25s/it]

[I 2026-06-04 16:34:47,854] Trial 1 pruned. 
Best trial score:
0.1024090857320655

Best params:
{'solver': 'saga', 'C': 80.79031041038265, 'l1_ratio': 0.7772266014546229, 'class_weight': None, 'fit_intercept': True, 'tol': 0.000375763511914987, 'max_iter': 3246}


In [22]:
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=30, n_jobs=-1, show_progress_bar=True)

Best trial: 21. Best value: 0.102408:   3%|████▌                                                                                                                                     | 1/30 [00:28<13:58, 28.92s/it]

[I 2026-06-04 16:44:45,983] Trial 33 finished with value: 0.10240996680785783 and parameters: {'solver': 'saga', 'C': 3.1604235239083516, 'l1_ratio': 0.43597493556829703, 'class_weight': None, 'fit_intercept': True, 'tol': 0.0029844473779650337, 'max_iter': 4929}. Best is trial 21 with value: 0.1024080564765762.


Best trial: 21. Best value: 0.102408:   7%|█████████▏                                                                                                                                | 2/30 [00:29<05:36, 12.03s/it]

[I 2026-06-04 16:44:46,187] Trial 35 finished with value: 0.10240808052403856 and parameters: {'solver': 'saga', 'C': 3.5068749000188606, 'l1_ratio': 0.3956115365538211, 'class_weight': None, 'fit_intercept': True, 'tol': 0.0030773253875507438, 'max_iter': 3977}. Best is trial 21 with value: 0.1024080564765762.


Best trial: 21. Best value: 0.102408:  10%|█████████████▊                                                                                                                            | 3/30 [00:30<03:16,  7.29s/it]

...

[I 2026-06-04 16:46:10,153] Trial 56 finished with value: 0.10240794640363877 and parameters: {'solver': 'saga', 'C': 1.248725444175947, 'l1_ratio': 0.3763498868060859, 'class_weight': None, 'fit_intercept': True, 'tol': 0.00019162023473349643, 'max_iter': 4090}. Best is trial 51 with value: 0.10240530596157663.


Best trial: 51. Best value: 0.102405:  97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 29/30 [02:26<00:14, 14.09s/it]

[I 2026-06-04 16:46:43,453] Trial 34 finished with value: 0.10240839475615535 and parameters: {'solver': 'saga', 'C': 4.735435152940276, 'l1_ratio': 0.4482300101988604, 'class_weight': None, 'fit_intercept': True, 'tol': 0.00019965845212452694, 'max_iter': 4943}. Best is trial 51 with value: 0.10240530596157663.


Best trial: 51. Best value: 0.102405: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [02:45<00:00,  5.50s/it]

[I 2026-06-04 16:47:02,075] Trial 54 finished with value: 0.10240893293505852 and parameters: {'solver': 'saga', 'C': 19.1083000578225, 'l1_ratio': 0.3370590670101929, 'class_weight': None, 'fit_intercept': True, 'tol': 0.00026240000267621573, 'max_iter': 3312}. Best is trial 51 with value: 0.10240530596157663.


In [23]:
lg = LogisticRegression(**study.best_params).fit(X_train, y_train.class_encoded)
train_proba = cross_val_predict(lg, X_train, y_train.class_encoded, cv=StratifiedKFold(shuffle=True, random_state=42, n_splits=5), n_jobs=-1, method='predict_proba')

In [24]:
test_proba = lg.predict_proba(X_test)

In [25]:
optimizer = MulticlassThresholdOptimizer()
optimizer.fit(train_proba, y_train.class_encoded)

,n_splits,5
,method,'Nelder-Mead'
,maxiter,500
,random_state,42


In [26]:
final_predictions_idx = optimizer.predict(test_proba)

class_mapping = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
sub_labels = [class_mapping[idx] for idx in final_predictions_idx]

# Submission

In [27]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lg.csv', index=False)

In [28]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [29]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'cat_0', 'cat_1', 'xgb_0', 'xgb_1', 'hist_0',
       'hist_1', 'extra_0', 'extra_1', 'rf_0', 'rf_1'],
      dtype='str')